# StatsBomb FIFA Women's passing network

In [15]:
import pandas as pd
import numpy as np
from tqdm import tqdm

## Core algorithm we've seen in the sample

In [22]:
# Load the trained xT grid 
xT = pd.read_csv('//Users/shawnhan/Desktop/Singh_xT_Grid.csv', header=None)
xT = np.array(xT)
xT_rows, xT_cols = xT.shape

def calculate_xt_for_events(df):
    """
    Calculates the xT value for the location of all events in a DataFrame.
    This is designed to be run on the raw events DataFrame.
    """
    df_copy = df.copy()

    # Extract coordinates if they exist
    df_copy['x_coord'] = df_copy['location'].apply(lambda x: x[0] if isinstance(x, list) and len(x) >= 2 else None)
    df_copy['y_coord'] = df_copy['location'].apply(lambda x: x[1] if isinstance(x, list) and len(x) >= 2 else None)
    
    # Scale coordinates
    df_copy['x_scaled'] = (df_copy['x_coord'] / 120) * 105
    df_copy['y_scaled'] = (df_copy['y_coord'] / 80) * 68
    
    # Bin coordinates into xT grid cells
    df_copy['x_bin'] = pd.cut(df_copy['x_scaled'], bins=xT_cols, labels=False, include_lowest=True)
    df_copy['y_bin'] = pd.cut(df_copy['y_scaled'], bins=xT_rows, labels=False, include_lowest=True)
    
    # Lookup the xT value for each event's location
    def lookup_xt(x_bin, y_bin):
        if pd.notna(x_bin) and pd.notna(y_bin):
            return xT[int(y_bin), int(x_bin)]  
        return 0
    
    df_copy['xT_value'] = df_copy.apply(lambda row: lookup_xt(row['x_bin'], row['y_bin']), axis=1)
    
    return df_copy

def build_weighted_passing_network_robust(df_with_xt):
    """
    Builds the passing network by linking 'Pass' events to subsequent 'Ball Receipt*' events.
    This is the robust method that correctly identifies passers and receivers.
    """
    df = df_with_xt.copy()
    
    df['player_name'] = df['player'].apply(lambda x: x.get('name', '') if isinstance(x, dict) else '')
    df['team_name'] = df['team'].apply(lambda x: x.get('name', '') if isinstance(x, dict) else '')
    df['event_type'] = df['type'].apply(lambda x: x.get('name', '') if isinstance(x, dict) else '')
    
    df = df.sort_values(['minute', 'second']).reset_index(drop=True)
    
    pass_connections = []
    
    for team_name in df['team_name'].unique():
        if not team_name: continue
        team_data = df[df['team_name'] == team_name].reset_index(drop=True)
        
        i = 0
        while i < len(team_data) - 1:
            current_event = team_data.iloc[i]
            
            if current_event['event_type'] == 'Pass':
                j = i + 1
                while j < len(team_data):
                    next_event = team_data.iloc[j]
                    
                    if next_event['event_type'] == 'Ball Receipt*':
                        passer = current_event['player_name']
                        receiver = next_event['player_name']
                        
                        if passer and receiver and passer != receiver:
                            xt_start = current_event['xT_value']
                            xt_end = next_event['xT_value']
                            xt_diff = xt_end - xt_start
                            
                            if xt_diff > 0:
                                base_weight = 1 + (xt_diff * 10)
                            else:
                                base_weight = max(0.1, 1 + (xt_diff * 5))
                            
                            pressure_bonus = 0.15 if current_event.get('under_pressure') else 0
                            
                            pass_connections.append({
                                'passer': passer,
                                'receiver': receiver,
                                'weight': base_weight + pressure_bonus
                            })
                        break
                    
                    elif next_event['event_type'] == 'Pass':
                        break
                    
                    j += 1
            i += 1
            
    if not pass_connections:
        return None, None, None

    passes_df = pd.DataFrame(pass_connections)
    
    all_players = sorted(list(set(passes_df['passer'].tolist() + passes_df['receiver'].tolist())))
    player_to_idx = {player: idx for idx, player in enumerate(all_players)}
    n_players = len(all_players)
    
    adjacency_matrix = np.zeros((n_players, n_players))
    
    for _, row in passes_df.iterrows():
        passer_idx = player_to_idx.get(row['passer'])
        receiver_idx = player_to_idx.get(row['receiver'])
        if passer_idx is not None and receiver_idx is not None:
            adjacency_matrix[passer_idx, receiver_idx] += row['weight']
            
    return adjacency_matrix, all_players, passes_df

def calculate_weighted_pagerank(weighted_adj_matrix, players, damping_factor=0.85, max_iterations=100, tolerance=1e-6):
    n = len(players)
    if n == 0: return np.array([])
    
    row_sums = weighted_adj_matrix.sum(axis=1, keepdims=True)
    with np.errstate(divide='ignore', invalid='ignore'):
        transition_matrix = np.nan_to_num(weighted_adj_matrix / row_sums)
    
    dangling_nodes = np.where(row_sums.flatten() == 0)[0]
    pagerank_scores = np.ones(n) / n
    
    for _ in range(max_iterations):
        old_scores = pagerank_scores.copy()
        new_scores = transition_matrix.T.dot(pagerank_scores)
        dangling_sum = np.sum(pagerank_scores[dangling_nodes])
        new_scores += dangling_sum / n
        pagerank_scores = (1 - damping_factor) / n + damping_factor * new_scores
        if np.linalg.norm(pagerank_scores - old_scores, 1) < tolerance:
            break
            
    return pagerank_scores

In [23]:
import os

events_directory = '/Users/shawnhan/Desktop/Pioneer/Pioneer_Final_Project/Download_Statsbomb/wwc2023/events/'

all_filenames = os.listdir(events_directory)

match_ids = [f.replace('.json', '') for f in all_filenames if f.endswith('.json')]

print(f"Found {len(match_ids)} match files to process.")

Found 64 match files to process.


## Batch processing

In [24]:
def process_single_match(match_id):
    try:
        # 1. Load the raw JSON data
        df = pd.read_json(f'/Users/shawnhan/Desktop/Pioneer/Pioneer_Final_Project/Download_Statsbomb/wwc2023/events/{match_id}.json')

        # 2. Calculate xT for all events
        df_with_xt = calculate_xt_for_events(df)
        
        # 3. Build the passing network
        weighted_adj, players, weighted_passes_df = build_weighted_passing_network_robust(df_with_xt)
        
        # 4. If no network, skip match
        if weighted_adj is None:
            return None
        
        # 5. Calculate PageRank
        scores = calculate_weighted_pagerank(weighted_adj, players)
        
        df['player_id'] = df['player'].apply(lambda x: x.get('id') if isinstance(x, dict) else None)
        df['player_name'] = df['player'].apply(lambda x: x.get('name') if isinstance(x, dict) else None)
        df['team_name'] = df['team'].apply(lambda x: x.get('name') if isinstance(x, dict) else None)
        
        info_df = df.dropna(subset=['player_id', 'player_name']).drop_duplicates(subset=['player_id'])
        
        player_info = {}
        for _, row in info_df.iterrows():
            player_info[row['player_name']] = {
                'team_id': row['team']['id'],
                'team': row['team_name'],
                'player_id': row['player_id']
            }

        results = []
        for i, player_name in enumerate(players):
            info = player_info.get(player_name, {})
            results.append({
                'match_id': match_id,
                'team_id': info.get('team_id'),
                'team': info.get('team'),
                'player_id': info.get('player_id'),
                'player': player_name,
                'WPR_score': scores[i]
            })
        
        df_results = pd.DataFrame(results)
        df_results = df_results.sort_values('WPR_score', ascending=False).reset_index(drop=True)
        df_results['match_rank'] = range(1, len(df_results) + 1)
        
        return df_results[['match_id', 'team_id', 'team', 'player_id', 'player', 'WPR_score', 'match_rank']]
        
    except Exception as e:
        import traceback
        print(f"Error processing match {match_id}: {str(e)}\n{traceback.format_exc()}")
        return None

all_results = []
failed_matches = []

for match_id in tqdm(match_ids, desc="Processing matches"):
    result = process_single_match(match_id)
    if result is not None:
        all_results.append(result)
    else:
        failed_matches.append(match_id)

print(f"\nSuccessfully processed: {len(all_results)} matches")
print(f"Failed matches: {len(failed_matches)}")

if all_results:
    combined_df = pd.concat(all_results, ignore_index=True)
    combined_df.to_csv('all_matches_wpr_combined.csv', index=False)
    print("\nSuccessfully saved combined results to 'all_matches_wpr_combined.csv'")

Processing matches: 100%|██████████| 64/64 [00:29<00:00,  2.17it/s]


Successfully processed: 64 matches
Failed matches: 0

Successfully saved combined results to 'all_matches_wpr_combined.csv'
